# Colab slice: concept-drift power (CF / RF / DRPerm)

Runs **one** board cell that finishes on free Colab CPU:

- dataset: `msrvtt_concept_top30` (synthetic multimodal, concept drift on `Y|X`)
- combo: `text_audio` only
- methods: **cfperm · rfperm · drperm**
- default: `n=200`, `B=30`, `n_perm=40` (~15–30 min)

Overnight reference (B=120, `n=500`, `n_perm≈40–150`) for the same cell:

| method | power@0.05 |
|---|---:|
| rfperm | 1.00 |
| cfperm | 0.60 |
| drperm | 0.40 |

Checkpointed: re-run the same cell to stack more reps.

In [ ]:
# 1) clone + deps (CPU is enough)
import os, sys
from pathlib import Path

REPO = "https://github.com/ruanhq-StatsML/SAPerm_.git"
BRANCH = "cursor/cfdr-accel-perm-parallel-e93c"  # or main when merged
ROOT = Path("/content/SAPerm")

if not (ROOT / ".git").exists():
    !git clone --depth 1 -b {BRANCH} {REPO} {ROOT}
else:
    %cd {ROOT}
    !git fetch origin {BRANCH} && git checkout {BRANCH} && git pull --ff-only origin {BRANCH}

%cd {ROOT}
!pip -q install numpy pandas scikit-learn joblib scipy

sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "benchmark_methods"))
print("cwd", Path.cwd())

In [ ]:
# 2) smoke: one rep should finish in ~20s
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["CFDR_PERM_JOBS"] = "2"

from experiments.multimodal_attribution.overnight_power import DatasetSpec, make_dataset_batch
from experiments.multimodal_attribution.power_benchmark import build_methods
from experiments.multimodal_attribution.parallel_eval import eval_methods_parallel

spec = DatasetSpec("msrvtt_concept_top30", "concept_drift", band="top30", concept_shift=0.35)
batch = make_dataset_batch(spec, ("text", "audio"), n=120, seed=0)
methods = build_methods(n_perm=12, include_po_interact=False, include_fsds=False, include_perm_suite=True)
keep = [(k, methods[k]) for k in ("cfperm", "rfperm", "drperm")]
X1, X2 = batch["df_ref"][:, :-1], batch["df_new"][:, :-1]
print(eval_methods_parallel(keep, X1, X2, batch, n_jobs=1))

In [ ]:
# 3) main Colab board (edit knobs if you want longer / shorter)
import os
os.environ["CFDR_PERM_JOBS"] = "2"

!python -m experiments.multimodal_attribution.colab_concept_power \
  --n 200 --n-rep 30 --n-perm 40 --n-jobs 2 \
  --out-dir experiments/multimodal_attribution/results/colab_concept

In [ ]:
# 4) show power table + compare to overnight B=120
from pathlib import Path
import pandas as pd

out = Path("experiments/multimodal_attribution/results/colab_concept")
raw = pd.read_csv(out / "power_raw.csv")
alpha = 0.05
tab = (
    raw.groupby(["dataset", "method"], as_index=False)
    .agg(B=("rep", "count"), power=("pvalue", lambda s: (s < alpha).mean()), mean_p=("pvalue", "mean"))
    .sort_values(["dataset", "method"])
)
display(tab)

ref = pd.DataFrame([
    {"method": "rfperm", "overnight_B120_power": 1.00},
    {"method": "cfperm", "overnight_B120_power": 0.60},
    {"method": "drperm", "overnight_B120_power": 0.40},
])
merged = tab.merge(ref, on="method", how="left")
display(merged)

board = out / "POWER_BOARD.md"
print(board.read_text() if board.exists() else "(no POWER_BOARD yet)")

### Optional: stack more reps / add null type-I

```bash
# raise B without redoing finished cells
python -m experiments.multimodal_attribution.colab_concept_power --n 200 --n-rep 60 --n-perm 40 --n-jobs 2

# also run null (type-I) on the same combo
python -m experiments.multimodal_attribution.colab_concept_power --n 200 --n-rep 30 --n-perm 40 --null --n-jobs 2
```

Full overnight boards stay in the cloud agent (`n=500`, `B=120`, all combos + classical/MSR).